In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier





# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

#df['S_3'].fillna(df['S_3'].mean(), inplace=True)


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"])

scaler = StandardScaler()
scaled = scaler.fit_transform(numerical_cols)
df.head()


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df):
  print("Target Distribution:")
  print(df.value_counts(normalize=True))
  sns.countplot(x=df)
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df['B_1'])

In [ ]:
# Task 1: Write your code here:
X = df.drop("B_1", axis=1).astype(float)
y = df['P_2'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

atboost_model = models["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 1: Write your code here:
X, y = make_regression(n_samples=1000, n_features=10, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest R²:", rf.score(X_test, y_test))


# Feature importance
print("\nTop 3 important features (Random Forest):")
importances = rf.feature_importances_
for i in np.argsort(importances)[-3:][::-1]:
    print(f"  Feature {i}: {importances[i]:.3f}")

In [ ]:
# Task 2: Write your code here:
sklearn_models = {
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: